# Lab 02: IPO Research Agent

## Business Context

The data is ready — filings are parsed, chunked, and indexed; stock performance is in a Delta table. Now build the research agent: the first version that can answer **"What did Snowflake say about competition, and how did SNOW perform?"**

This is the prototype the analysts will test. It combines unstructured text retrieval (Vector Search) with structured data lookup (UC SQL functions) in a single agent.

| Attribute | Detail |
|---|---|
| **Key Skills** | @tool decorator, UC SQL functions, UCFunctionToolkit, create_react_agent, multi-tool agents |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | ~$2-3 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph unitycatalog-ai[databricks] unitycatalog-langchain --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
LLM_ENDPOINT = "databricks-meta-llama-3.1-405b-instruct"
VS_ENDPOINT = "ipo_analyzer_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.filing_chunks_index"

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"LLM     : {LLM_ENDPOINT}")

## A. Build the Retrieval Tool

The retrieval tool wraps Vector Search — when the agent needs to find relevant passages from S-1 filings, it calls this tool. The `@tool` decorator turns any Python function into a tool the LLM can invoke.

`query_type="HYBRID"` combines semantic search (meaning) with keyword search (exact terms) via Reciprocal Rank Fusion — the recommended default for RAG.

In [ ]:
from databricks.vector_search.client import VectorSearchClient
from langchain_core.tools import tool

vsc = VectorSearchClient()
_vs_index = vsc.get_index(VS_ENDPOINT, VS_INDEX)

COMPANY_TICKERS = {
    'snowflake': 'SNOW', 'palantir': 'PLTR', 'doordash': 'DASH',
    'coinbase': 'COIN', 'rivian': 'RIVN', 'unity': 'U',
    'roblox': 'RBLX', 'bumble': 'BMBL', 'affirm': 'AFRM',
    'robinhood': 'HOOD', 'toast': 'TOST', 'confluent': 'CFLT',
    'gitlab': 'GTLB', 'hashicorp': 'HCP', 'duolingo': 'DUOL',
    'instacart': 'CART', 'klaviyo': 'KVYO', 'arm': 'ARM',
    'reddit': 'RDDT', 'rubrik': 'RBRK', 'astera': 'ALAB',
    'snow': 'SNOW', 'pltr': 'PLTR', 'dash': 'DASH', 'coin': 'COIN',
    'rivn': 'RIVN', 'rblx': 'RBLX', 'bmbl': 'BMBL', 'afrm': 'AFRM',
    'hood': 'HOOD', 'tost': 'TOST', 'cflt': 'CFLT', 'gtlb': 'GTLB',
    'hcp': 'HCP', 'duol': 'DUOL', 'cart': 'CART', 'kvyo': 'KVYO',
    'rddt': 'RDDT', 'rbrk': 'RBRK', 'alab': 'ALAB',
}

def _extract_tickers(query):
    found = []
    q_lower = query.lower()
    for name, ticker in COMPANY_TICKERS.items():
        if name in q_lower and ticker not in found:
            found.append(ticker)
    return found

def _extract_keywords(query):
    stopwords = {'what', 'did', 'say', 'about', 'in', 'their', 'the', 'and', 'how',
                 'does', 'do', 'compare', 'between', 'from', 'filing', 'filings',
                 's-1', 's1', 'ipo', 'for', 'with', 'that', 'this', 'are', 'was',
                 'were', 'has', 'have', 'had', 'been', 'their', 'they', 'its'}
    company_words = set(COMPANY_TICKERS.keys())
    words = query.lower().replace('?', '').replace(',', '').replace('.', '').split()
    return [w for w in words if w not in stopwords and w not in company_words and len(w) > 2][:5]

@tool
def search_filings(query: str) -> str:
    """Search S-1 filing text for passages relevant to the query.
    Use this for questions about what companies said in their IPO filings.
    Include the company name in your query for best results."""
    tickers = _extract_tickers(query)
    keywords = _extract_keywords(query)
    all_parts = []
    seen_texts = set()

    if tickers:
        for ticker in tickers[:2]:
            # Stage 1: SQL keyword search for precise matches
            if keywords:
                like_clauses = ' OR '.join(
                    [f"LOWER(chunk_text) LIKE '%{kw}%'" for kw in keywords]
                )
                sql = f"""
                    SELECT chunk_text, path FROM {CATALOG}.{SCHEMA}.filing_chunks
                    WHERE LOWER(path) LIKE '%{ticker}%'
                      AND ({like_clauses})
                    ORDER BY chunk_index LIMIT 10
                """
                try:
                    for row in spark.sql(sql).collect():
                        text_key = row.chunk_text[:100]
                        if text_key not in seen_texts:
                            seen_texts.add(text_key)
                            source = row.path.split('/')[-1].replace('-S1.html', '')
                            all_parts.append(f'[Source: {source}]\n{row.chunk_text}')
                except Exception:
                    pass

            # Stage 2: Vector search for semantic matches
            results = _vs_index.similarity_search(
                query_text=query, columns=['chunk_text', 'path'],
                num_results=10, filters={'path LIKE': f'%{ticker}%'},
                query_type='HYBRID',
            )
            for doc in results.get('result', {}).get('data_array', []):
                text_key = doc[0][:100]
                if text_key not in seen_texts:
                    seen_texts.add(text_key)
                    source = doc[1].split('/')[-1].replace('-S1.html', '')
                    all_parts.append(f'[Source: {source}]\n{doc[0]}')
    else:
        results = _vs_index.similarity_search(
            query_text=query, columns=['chunk_text', 'path'],
            num_results=15, query_type='HYBRID',
        )
        for doc in results.get('result', {}).get('data_array', []):
            source = doc[1].split('/')[-1].replace('-S1.html', '')
            all_parts.append(f'[Source: {source}]\n{doc[0]}')

    return '\n\n---\n\n'.join(all_parts[:20]) if all_parts else 'No relevant passages found.'


# Smoke test
print(search_filings.invoke("Snowflake competitive advantages")[:500])

## B. Create UC Functions as Tools

Unity Catalog functions are governed, reusable SQL or Python functions stored in the catalog. When exposed as agent tools via `UCFunctionToolkit`, the LLM can call them like any other tool.

Two functions:
- **`get_filing_metadata`** — How many chunks do we have for a given company? (queries the chunks table)
- **`get_stock_performance`** — What was the stock's 12-month return? (queries the stock table)

> **Why SQL?** These are simple lookups over Delta tables. SQL UDFs have no cold-start overhead and are natively optimized by Spark. The `COMMENT` field becomes the tool description the LLM reads to decide when to call it.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_filing_metadata(company_name STRING)
RETURNS TABLE (path STRING, chunk_count BIGINT)
COMMENT 'Return the filing path and total chunk count for a company. Use this to check what filings are available and how much content each contains. Search by company name (e.g., Snowflake, DoorDash).'
RETURN
  SELECT path, COUNT(*) AS chunk_count
  FROM {CATALOG}.{SCHEMA}.filing_chunks
  WHERE LOWER(path) LIKE CONCAT('%', LOWER(company_name), '%')
  GROUP BY path
  ORDER BY chunk_count DESC
""")

print(f"Created: {CATALOG}.{SCHEMA}.get_filing_metadata")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.get_filing_metadata('snowflake')"))

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_stock_performance(ticker_symbol STRING)
RETURNS TABLE (company STRING, ticker STRING, sector STRING, ipo_date STRING,
               ipo_price DOUBLE, price_3m DOUBLE, price_6m DOUBLE, price_12m DOUBLE,
               twelve_month_return_pct DOUBLE)
COMMENT 'Look up post-IPO stock performance for a company by ticker symbol (e.g., SNOW, DASH, COIN). Returns IPO price, prices at 3/6/12 months, and percentage return.'
RETURN
  SELECT company, ticker, sector, ipo_date, ipo_price, price_3m, price_6m, price_12m, twelve_month_return_pct
  FROM {CATALOG}.{SCHEMA}.stock_performance
  WHERE UPPER(ticker) = UPPER(ticker_symbol)
""")

print(f"Created: {CATALOG}.{SCHEMA}.get_stock_performance")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.get_stock_performance('SNOW')"))

# ── Python UDF alternative (equivalent but with cold-start overhead): ─────
# spark.sql(f"""
# CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_stock_performance(ticker_symbol STRING)
# RETURNS STRING
# LANGUAGE PYTHON
# COMMENT 'Look up post-IPO stock performance by ticker.'
# AS $$
#     # Python UDFs have ~1-2s cold start. SQL is preferred for simple lookups.
#     return f"Stock lookup for {{ticker_symbol}}"
# $$
# """)

## C. Build the Multi-Tool Agent

Now we combine all three tools in a single **LangGraph ReAct agent**:

1. **`search_filings`** — Vector Search retrieval (from Section A)
2. **`get_filing_metadata`** — UC SQL function via `UCFunctionToolkit`
3. **`get_stock_performance`** — UC SQL function via `UCFunctionToolkit`

`UCFunctionToolkit` reads the UC function schemas (name, `COMMENT`, parameter types) and builds LangChain-compatible tool objects automatically. The LLM selects the appropriate tool(s) based on the question.

`create_react_agent` (LangGraph) manages the reasoning loop: the LLM decides which tool to call, LangGraph executes it, feeds the result back, and repeats until the LLM has enough information to answer.

In [ ]:
from langchain_community.chat_models import ChatDatabricks
from langgraph.prebuilt import create_react_agent
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

# ── LLM ──────────────────────────────────────────────────────────────────
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=1024, temperature=0.1)

# ── UC function tools ────────────────────────────────────────────────────
client = DatabricksFunctionClient()
uc_toolkit = UCFunctionToolkit(
    function_names=[
        f"{CATALOG}.{SCHEMA}.get_filing_metadata",
        f"{CATALOG}.{SCHEMA}.get_stock_performance",
    ],
    client=client,
)

# ── Combine all tools ────────────────────────────────────────────────────
all_tools = [search_filings] + uc_toolkit.tools
print(f"Tools: {[t.name for t in all_tools]}")

# ── System prompt ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an IPO Filing Analyzer for a financial research firm. "
    "You have access to S-1 filings from tech IPOs and stock performance data.\n\n"
    "- search_filings: Search S-1 text for relevant passages\n"
    "- get_filing_metadata: Look up chunk count and filing info for a company\n"
    "- get_stock_performance: Look up stock returns by ticker symbol\n\n"
    "Always cite the source filing when answering research questions. "
    "When asked about stock performance, use get_stock_performance with the ticker.\n\n"
    "IMPORTANT: You provide financial ANALYSIS, not investment ADVICE. "
    "Never recommend buying or selling stocks."
)

# ── Agent ────────────────────────────────────────────────────────────────
agent = create_react_agent(llm, all_tools, prompt=SYSTEM_PROMPT)
print("Agent ready.")

In [ ]:
# THE BUSINESS QUERY: combines filing text + stock data
query = "What did Snowflake say about competition in their S-1, and how did SNOW perform in its first year?"

print(f"Q: {query}")
print("=" * 70)
result = agent.invoke({"messages": [{"role": "user", "content": query}]})
print(result["messages"][-1].content)

In [ ]:
query2 = "Compare what Coinbase and Robinhood say about regulatory risk in their S-1 filings."

print(f"Q: {query2}")
print("=" * 70)
result2 = agent.invoke({"messages": [{"role": "user", "content": query2}]})
print(result2["messages"][-1].content)

In [ ]:
query3 = "How many chunks do we have for DoorDash, and what was DASH's 12-month stock return?"

print(f"Q: {query3}")
print("=" * 70)
result3 = agent.invoke({"messages": [{"role": "user", "content": query3}]})
print(result3["messages"][-1].content)

## Before / After

**Before this lab:** You had searchable chunks and a stock table, but no way to ask questions that span both. Getting "what did Snowflake say about competition AND how did the stock do?" required manually querying two separate systems.

**After this lab:** One agent, one question, one answer — grounded in the S-1 filing with citations, enriched with stock performance data.

In [ ]:
# BEFORE: Raw passages without synthesis
print("=== BEFORE: Raw passages (no agent) ===")
raw = search_filings.invoke("Snowflake competition")
print(raw[:400])
print("\n(Raw passages only — no synthesis, no stock data)\n")

# AFTER: Agent-synthesised answer with citations + stock data
print("=== AFTER: Agent answer ===")
result = agent.invoke({"messages": [{"role": "user", "content": "Briefly: what does Snowflake say about competition and how did SNOW perform?"}]})
print(result["messages"][-1].content)

In [ ]:
# Return key results via notebook exit (enables API result inspection)
import json as _json
_results = {}
try:
    _results["q1_snowflake"] = result["messages"][-1].content[:500]
    _results["q2_compare"] = result2["messages"][-1].content[:500]
    _results["q3_doordash"] = result3["messages"][-1].content[:500]
except Exception as e:
    _results["error"] = str(e)[:300]
dbutils.notebook.exit(_json.dumps(_results))


In [ ]:
# Scorecard -- tracks cumulative progress
try:
    import sys
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _parent = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
    if _parent not in sys.path:
        sys.path.insert(0, _parent)
    from shared.lab_utils import get_scorecard
    get_scorecard()
except Exception as e:
    print(f"Scorecard skipped: {e}")